# Technologies.csv Generation — Projection 2050 — Norte Amazónica

Generates one `Technologies.csv` per cluster (C1–C5) **per scenario** for EnergyScope ESMC,
**2050 projection**, copied from `analyse data ramp/reality/technologies.ipynb` (2025 reality
scenario) with four mandatory corrections for the projection horizon plus a vintage-aware
existing-fleet chaining rule (rule (e) below).

Base: sufficiency output (`../sufficiency/output_energyscope/C{k}/Technologies.csv`) — the same 2025
base used by the reality scenario, since no sufficiency scenario exists for 2035/2050 yet. Structural
rows (LED lighting, stove capacity factor, storage technology list, etc.) are assumed unchanged.
Only values documented below are overwritten; all other rows are inherited unchanged from sufficiency.

**Cost/lifetime block:** `c_inv`, `c_maint`, `gwp_constr`, `lifetime` (plus `Category`, `Subcategory`,
`Technologies name`, `Comment`) are **not** produced by the sufficiency base above — that file only
carries the 8 capacity columns (`c_p`, `fmin_perc`, `fmax_perc`, `f_min`, `f_max`, `f_min_prod`,
`f_max_prod`). Those 8 columns are merged in (rule (f) below) from the deployed, calibrated 2025
reality catalog.

**Rules applied:**
- **(a) Electricity generators** — `f_max = f_min` (locks capacity to existing installed amounts; no
  new construction), **excluding `PV_HS`/`HS_DIESEL`** (see note below).
- **(b) Off-grid SHS kits** — `PV_HS` / `HS_DIESEL`: `f_min = 0` (the 2012-vintage legacy kits are past
  their catalogue lifetime — PV_HS 20y, HS_DIESEL 5y — by this horizon), but `f_max` is left
  **unconstrained** (inherited from sufficiency, `1e15`) so the model can build new SHS capacity.
- **(c) Storage** — `f_max = f_min` for all battery/storage technologies **except `BATT_HS`**, which
  keeps the sufficiency base's own values (`f_min = 0`, `f_max = 1e15`, also past its 10y catalogue
  lifetime).
- **(d) Stoves** — `f_min` recalculated from Census 2024 cooking fuel data (**not** projected to this
  horizon — see "Missing entries" at the end of this notebook); `f_max` unchanged from sufficiency.
  Runs *after* rule (e) below and always wins for `STOVE_WOOD`/`STOVE_LPG` — same ordering as every
  prior build of this notebook.
- **(e) Vintage-aware capacity chaining from 2035 (REPLACES the previous "existing fleet forced to
  0 at 2050" rule)** — see the dedicated markdown cell right before Section 4 for the full decision
  record. In short: `f_min[2050,c]` is chained from each scenario's own **solved** 2035 case-study
  output (`Assets.csv`, column `F`), not from a hand-picked constant. This makes rule (e), for the
  first time, **scenario-specific**: `capacity_only`/`assembled` below are keyed by
  `(scenario, cluster)` instead of just `cluster`, and every downstream section (5 through 9) loops
  over `DEPLOY_SCENARIOS_2050` accordingly.
- **(f) Cost/lifetime block** — `c_inv`, `c_maint`, `gwp_constr`, `lifetime` (plus the
  `Category`/`Subcategory`/`Technologies name`/`Comment` structural columns) are merged in verbatim
  from `Data/2025/reality/02_REF_REGION/Technologies.csv`, the deployed and calibrated Norte Amazónica
  2025 catalog. A blocking assertion (Section 5) checks these four columns are byte-identical to that
  reference for every technology in every scenario/cluster, with **no exceptions** — cost trajectories
  for this horizon are not modelled yet. The notebook halts before writing any file if the assertion
  fails.
- **(g) `DEC_SOLAR.f_max`** — inherited from the deployed `reality_access` reference (see Section 1
  setup cell for the full rationale).
- **(h) `PV_HS`/`HS_DIESEL` `f_max_prod`** — capped at each cluster's dispersed demand (see Section 1
  setup cell).

**Consequence of rule (b), inherited from the 2035 build:** `Layers_in_out.csv` flags `PV_HS`/
`HS_DIESEL` as `ELECTRICITY` generators (54 generators here, not 52) — Rule (a)'s mask excludes
`OFF_GRID_TECHS` so it doesn't re-lock them before rule (b) runs.


In [1]:
import os
import pandas as pd

DEPLOY_SCENARIOS_2050 = ["no_transition", "early_access", "late_access", "early_access_brazil"]
OUT_DIR_BY_SCENARIO = {s: os.path.join("output_energyscope_2050", s) for s in DEPLOY_SCENARIOS_2050}
SUFF_DIR = "../sufficiency/output_energyscope"

# Rule (f): cost/lifetime block source -- the deployed, calibrated Norte Amazónica 2025 catalog.
# The sufficiency base above only carries capacity columns; c_inv/c_maint/gwp_constr/lifetime are
# merged in from here (Section 5), not projected in any way for this horizon.
COST_REF_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/02_REF_REGION/Technologies.csv"

# NREL ATB 2024, Moderate/Mid cost trajectory. Dimensionless ratio applied to the 2025
# reality catalog's c_inv (never an absolute substitution) -- immune to the catalog's mixed
# cost-year vintages (TS_DEC in 2015 EUR, PV_HS/BATT_HS in 2024 EUR, DEC_SOLAR undated).
# Same table as analyse data ramp/2035/technologies.ipynb -- kept in sync manually, both
# notebooks assert their own year's ratio exactly, so any drift is caught at execution time.
NREL_ATB_C_INV_RATIO = {
    "PV_UTILITY": {2035: 0.600, 2050: 0.458},
    "BATT_LI":    {2035: 0.757, 2050: 0.555},
    "PV_HS":      {2035: 0.65,  2050: 0.46},
    "BATT_HS":    {2035: 0.76,  2050: 0.61},
}

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# Rule (g): DEC_SOLAR.f_max -- the sufficiency base above is stale on this column (local
# ../sufficiency/output_energyscope/C{k}/Technologies.csv carries f_max=0, but the actually
# deployed Data/2025/sufficiency/C{k}/Technologies.csv has f_max=1e15 for every cluster).
# Separately, inheriting from Data/2025/reality would also be wrong here: reality/reality_phase2
# deliberately lock DEC_SOLAR (Source A households do not own a solar water heater today), but
# Sources B and C get the sufficiency bundle in every projection scenario, and sufficiency (like
# reality_access) leaves DEC_SOLAR investment open. Inherit explicitly, per cluster, from the
# deployed reality_access reference instead of trusting either the stale local base or reality.
DEC_SOLAR_FMAX_REF_PATH_TMPL = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality_access/C{k}/Technologies.csv"
DEC_SOLAR_FMAX_BY_CLUSTER = {}
for k in range(1, 6):
    ra = pd.read_csv(DEC_SOLAR_FMAX_REF_PATH_TMPL.format(k=k), sep=";", index_col=0)
    ra.index = ra.index.str.strip()
    DEC_SOLAR_FMAX_BY_CLUSTER[k] = float(ra.loc["DEC_SOLAR", "f_max"])
print("DEC_SOLAR f_max by cluster, from the deployed 2025 reality_access reference:", DEC_SOLAR_FMAX_BY_CLUSTER)

# Rule (h): PV_HS / HS_DIESEL f_max_prod -- absolute annual production ceiling [GWh/y],
# independent of f_max (capacity). ESMC_model_AMPL.mod only generates the f_max_prod_abs
# constraint when f_max_prod[c,j] < 1e14, so at the sufficiency base's inactive default (1e15)
# the constraint row does not exist at all. Cap each of PV_HS and HS_DIESEL (independently) at
# the cluster's dispersed demand. f_min_prod stays at its default 0 -- do NOT pin
# f_min_prod = f_max_prod, that breaks the solver.
DISPERSED_DEMAND_REF_PATH = "../../analyse_GIS_phase2_projections/output/2050/cluster_summary.csv"
_cluster_summary = pd.read_csv(DISPERSED_DEMAND_REF_PATH, index_col="Cluster")
DISPERSED_DEMAND_GWH_BY_CLUSTER = {
    k: float(_cluster_summary.loc[f"C{k}", "demande_dispersee_GWh"]) for k in range(1, 6)
}
print("Dispersed demand by cluster [GWh/y], from cluster_summary.csv:", DISPERSED_DEMAND_GWH_BY_CLUSTER)

# Load sufficiency output as base -- only modified rows are overwritten
suff = {}
for k in range(1, 6):
    path = os.path.join(SUFF_DIR, f"C{k}", "Technologies.csv")
    df = pd.read_csv(path, sep=";")
    df["Technologies param"] = df["Technologies param"].str.strip()
    suff[k] = df
print("Loaded sufficiency base for C1-C5")


DEC_SOLAR f_max by cluster, from the deployed 2025 reality_access reference: {1: 1000000000000000.0, 2: 1000000000000000.0, 3: 1000000000000000.0, 4: 1000000000000000.0, 5: 1000000000000000.0}
Dispersed demand by cluster [GWh/y], from cluster_summary.csv: {1: 2.0786, 2: 0.1321, 3: 0.6478, 4: 1.5431, 5: 0.0}
Loaded sufficiency base for C1-C5


## 0bis. Rule (e) — vintage-aware capacity chaining from 2035 (decision record)

The previous build of this notebook forced `GENSET_DIESEL`/`PV_UTILITY` `f_min = 0` in every
cluster at 2050, treating the entire 2025 fleet (diesel gensets + utility PV) as retired by this
horizon regardless of what actually got built along the way at 2035. That throws away real
information: `early_access`/`late_access` unlock `PV_UTILITY` investment at 2035
(`scripts/run.py`, `PV_UTILITY.f_max = 1e15`), and whatever got built there does not vanish
between 2035 and 2050 just because the *original* 2025 fleet is beyond its service life.

**Decision (vintage-aware register, legacy vs. new-build kept separate):** the pre-2025 legacy
share and the 2035 new-build share of installed capacity are chained independently.

- **`PV_UTILITY` / `GENSET_DIESEL`** — `f_min[2050,c] = F[2035,c] − legacy_floor[2035,c]`, floored
  at 0, where `legacy_floor` is the pre-existing 2025 fleet locked into 2035's own `Technologies.csv`
  (`EXISTING_FLEET_FMIN_GW` in `2035/technologies.ipynb`): `PV_UTILITY` 0.00040 (C4) / 0.00510 (C5);
  `GENSET_DIESEL` 0.05333 (C3) / 0.00742 (C4) / 0.02736 (C5). The legacy share is retired at 2050 (not
  chained); the 2035-vintage new-build share survives and becomes the new floor. `GENSET_DIESEL`
  was never unlocked past its legacy floor at 2035 in any scenario (`F[2035,c] == legacy_floor`
  everywhere), so its `f_min[2050,c]` is 0 in every scenario/cluster — not just `no_transition`.
  `PV_UTILITY` is unlocked in `early_access`/`late_access` only, so only those two scenarios get a
  nonzero chained floor.
- **All other surviving technologies** — plain passthrough, `f_min[2050,c] = F[2035,c]` from that
  scenario's own solved 2035 output (`Assets.csv`, column `F`; missing/blank treated as 0).
- **Non-surviving technologies** — `f_min[2050,c] = 0`. A technology's 2035-vintage capacity does
  not survive to 2050 when `2035 + lifetime <= 2050` (i.e. `lifetime <= 15`, boundary case included —
  `BATT_LI` at exactly 15y retires). This does not block new construction at 2050: `PV_HS`/`HS_DIESEL`
  already get `f_min = 0` unconditionally from rule (b); `BATT_LI`'s `f_max` is reopened at solve time
  for `early_access`/`late_access` by `scripts/run.py`, same override already used at 2035.
  Full list: `BATT_LI` (15y), `BATT_HS` (10y), `HS_DIESEL` (5y), `REFRIGERATOR_EL` (15y),
  `DEC_DIRECT_ELEC` (15y), `LED_BULB`/`LED_LIGHT` (10y), all `STOVE_*` (10y — see rule (d), which
  overrides `STOVE_WOOD`/`STOVE_LPG` afterward regardless), `COMM_MACHINERY_EL`/`FISH_MACHINERY_EL`
  (12y). Their diesel-fuelled counterparts (`COMM_MACHINERY_DIESEL`, `FISH_MACHINERY_DIESEL`, 25y)
  and `AGR_MACHINERY_EL`/`MIN_MACHINERY_EL` (12y/17y — never built with nonzero `F[2035,c]` in any
  scenario here, so moot) are *not* in this list; they fall under the generic passthrough rule above.
- **`EFFICIENCY`** — excluded from the chain entirely (not a capacity; it is a technical scaling
  parameter, reported as `F ≈ 0.909` in `Assets.csv` for structural reasons unrelated to installed
  GW). Left untouched by rule (e).
- **Fuel storage buffers** (`DIESEL_STORAGE`, `GASOLINE_STORAGE`, `JET_FUEL_STORAGE`, `LPG_STORAGE`,
  20y lifetime) are *not* electrochemical/home-battery storage — they survive and chain like any
  other generic technology under the passthrough rule above.

**Scenario sourcing:** each 2050 scenario chains from the matching 2035 scenario's own solve
(`no_transition`←`no_transition_2035`, `early_access`←`early_access_2035`,
`late_access`←`late_access_2035`). `early_access_brazil` has no 2035 lineage of its own (`Data/2035`
only has `no_transition`/`early_access`/`late_access`) — it chains from `early_access_2035`, same
supply-side trajectory, differing only at 2050 by the Brazil grid interconnection. This is an
assumption, flagged explicitly in the registry (`source_2035_scenario` column) rather than silently
baked in.

**Read this as a sequencing assumption, not integrated planning.** Because `early_access_brazil`
inherits `early_access_2035`'s chained capacity row for row (verified byte-identical, 150/150
rows, 2026-08-06), its pre-2035 investment decisions are, by construction, the *same* decisions
`early_access` made with no Brazil interconnection in its investment horizon at all. The model does
not let 2035-era investment anticipate or respond to a border line that, in this scenario, only
exists from 2050 onward. That is consistent with the real-world fact that the interconnection does
not exist today, but it is a modelling assumption, not a result: it means "the interconnection
arrives after 2035, unanticipated by anyone's prior investment," not "the interconnection was
planned for." State it as such wherever `early_access_brazil` is discussed — a reader who is not
shown this explicitly could otherwise read the shared 2035 lineage as evidence of coordinated
cross-border planning that isn't actually in the model.

**Registry:** every chained/retired row (technology, cluster, scenario, vintage year, capacity,
`c_inv`) is recorded in `REGISTRY` (built in the setup cell below) for later system-cost
reconstruction post-processing. `REGISTRY` and the deployed `f_min` values are printed/asserted in
Section 10-11 near the end of this notebook, and are what should be reported before any 2050 solve
is attempted.

**Known model behaviour, not an anomaly (reported here, not acted on):** in the 2035 solves this
chains from, `PV_HS`/`BATT_HS` ratio sits at exactly 2.0 in every active cluster — the
`pv_battery_ratio_max` ceiling. The model would prefer less battery per panel than the ceiling
currently allows; left as a modelling note for a future session, not touched here.


In [2]:
CS_2035_ROOT = "../../../EnergyScope_BO_nord_amazonia/case_studies/C1_C2_C3_C4_C5"

# Each 2050 scenario chains from the matching 2035 scenario's OWN solved output.
# early_access_brazil has no 2035 lineage of its own -- chains from early_access_2035
# (see decision-record markdown above).
SOURCE_2035_BY_2050_SCENARIO = {
    "no_transition": "no_transition",
    "early_access": "early_access",
    "late_access": "late_access",
    "early_access_brazil": "early_access",
}

# Utility-scale generation fleet: legacy (pre-2035) floor subtracted out, only the 2035-vintage
# new-build increment (if any) is chained to 2050. Values match EXISTING_FLEET_FMIN_GW in
# 2035/technologies.ipynb -- the floor locked into 2035's own Technologies.csv.
LEGACY_FLOOR_2035_GW = {
    "GENSET_DIESEL": {3: 0.05333, 4: 0.00742, 5: 0.02736},
    "PV_UTILITY":    {4: 0.00040, 5: 0.00510},
}

# 2035-vintage capacity does NOT survive to 2050: 2035 + lifetime <= 2050 (lifetime <= 15,
# boundary case included). See decision-record markdown above for the full list + rationale.
NON_SURVIVOR_TECHS_2050 = [
    "BATT_LI", "BATT_HS", "HS_DIESEL", "REFRIGERATOR_EL", "DEC_DIRECT_ELEC",
    "LED_BULB", "LED_LIGHT",
    "STOVE_WOOD", "STOVE_LPG", "STOVE_NG", "STOVE_OIL", "STOVE_ELEC",
    "COMM_MACHINERY_EL", "FISH_MACHINERY_EL",
]

# Not a capacity -- excluded from the chain entirely, left untouched by rule (e).
EXCLUDED_FROM_CHAINING = ["EFFICIENCY"]

# Fuel storage buffers (20y lifetime): not electrochemical/home-battery storage, survive and
# chain like any other generic technology (blanket passthrough below). Listed here only for
# documentation / registry tagging, not used programmatically.
FUEL_STORAGE_BUFFERS = ["DIESEL_STORAGE", "GASOLINE_STORAGE", "JET_FUEL_STORAGE", "LPG_STORAGE"]

SPECIAL_LEGACY_TECHS = set(LEGACY_FLOOR_2035_GW.keys())

# Cost/lifetime reference, needed here to enumerate the full technology catalogue and to record
# c_inv per vintage in the registry (rule (f) below re-reads/re-merges the same file in full).
_cost_ref_for_registry = pd.read_csv(COST_REF_PATH, sep=";", skiprows=[1])
_cost_ref_for_registry["Technologies param"] = _cost_ref_for_registry["Technologies param"].str.strip()
_cost_ref_for_registry = _cost_ref_for_registry.set_index("Technologies param")
ALL_CHAINABLE_TECHS = [t for t in _cost_ref_for_registry.index if t not in EXCLUDED_FROM_CHAINING]

CHAINED_FMIN_2050 = {}   # scenario -> {k -> {tech: f_min}}
REGISTRY_ROWS = []       # capacity / vintage_year / c_inv per (scenario, cluster, technology)

for scenario_2050, scenario_2035 in SOURCE_2035_BY_2050_SCENARIO.items():
    cs_dir_2035 = f"{CS_2035_ROOT}/norte_amazonia_{scenario_2035}_2035/outputs"

    # Always verify the 2035 source actually solved before trusting its Assets.csv -- the
    # pipeline writes normal-looking files even on a rejected solve.
    solve_info = pd.read_csv(f"{cs_dir_2035}/Solve_info.csv", sep=r"\t;\t", header=None,
                              index_col=0, engine="python")
    solve_result_num = int(float(solve_info.loc["solve_result_num", 1]))
    assert solve_result_num == 0, (
        f"{scenario_2035}_2035: solve_result_num={solve_result_num} != 0 -- "
        f"refusing to chain capacities from a non-optimal solve"
    )

    assets = pd.read_csv(f"{cs_dir_2035}/regional_results/Assets.csv", sep=";")
    assets["F"] = pd.to_numeric(assets["F"], errors="coerce").fillna(0.0)

    # Vintage-pricing fix: a vintage_year=2035 tranche must be priced at the actual 2035
    # catalog's c_inv (which now differs from 2025 for the NREL ATB technologies), not the flat
    # 2025 reference -- otherwise every chained_to_2050 row would silently under/over-state its
    # true construction-year cost. Cost is cluster-independent, so this is read once per
    # scenario_2035, outside the cluster loop. pre-2035 legacy rows still correctly use the flat
    # 2025 reference (that legacy genuinely dates from 2025 or earlier).
    cost_2035_path = (f"../../../EnergyScope_BO_nord_amazonia/Data/2035/{scenario_2035}/"
                       f"02_REF_REGION/Technologies.csv")
    cost_2035 = pd.read_csv(cost_2035_path, sep=";", skiprows=[1])
    cost_2035["Technologies param"] = cost_2035["Technologies param"].str.strip()
    cost_2035 = cost_2035.set_index("Technologies param")

    CHAINED_FMIN_2050[scenario_2050] = {}
    for k in range(1, 6):
        region = f"C{k}"
        f2035 = assets.loc[assets["Regions"] == region].set_index("Technologies")["F"]
        chained = {}

        for tech in ALL_CHAINABLE_TECHS:
            F = float(f2035.get(tech, 0.0))
            c_inv_pre2035 = float(_cost_ref_for_registry.loc[tech, "c_inv"])
            c_inv_2035 = float(cost_2035.loc[tech, "c_inv"])

            if tech in SPECIAL_LEGACY_TECHS:
                legacy = LEGACY_FLOOR_2035_GW[tech].get(k, 0.0)
                new_2035 = max(F - legacy, 0.0)
                if legacy > 0:
                    REGISTRY_ROWS.append(dict(scenario=scenario_2050, source_2035_scenario=scenario_2035,
                                               cluster=region, technology=tech,
                                               vintage_year="pre-2035", capacity_GW=legacy, c_inv=c_inv_pre2035,
                                               status="retired_at_2050", f_min_2050_GW=0.0))
                REGISTRY_ROWS.append(dict(scenario=scenario_2050, source_2035_scenario=scenario_2035,
                                           cluster=region, technology=tech,
                                           vintage_year=2035, capacity_GW=new_2035, c_inv=c_inv_2035,
                                           status="chained_to_2050", f_min_2050_GW=new_2035))
                chained[tech] = new_2035
            elif tech in NON_SURVIVOR_TECHS_2050:
                if F > 0:
                    REGISTRY_ROWS.append(dict(scenario=scenario_2050, source_2035_scenario=scenario_2035,
                                               cluster=region, technology=tech,
                                               vintage_year=2035, capacity_GW=F, c_inv=c_inv_2035,
                                               status="retired_at_2050", f_min_2050_GW=0.0))
                chained[tech] = 0.0
            else:
                if F > 0:
                    REGISTRY_ROWS.append(dict(scenario=scenario_2050, source_2035_scenario=scenario_2035,
                                               cluster=region, technology=tech,
                                               vintage_year=2035, capacity_GW=F, c_inv=c_inv_2035,
                                               status="chained_to_2050", f_min_2050_GW=F))
                chained[tech] = F

        CHAINED_FMIN_2050[scenario_2050][k] = chained

    print(f"{scenario_2050}_2050 chained from {scenario_2035}_2035 (solve_result_num={solve_result_num})")

REGISTRY = pd.DataFrame(REGISTRY_ROWS)
print()
print(f"Vintage registry: {len(REGISTRY)} rows "
      f"({(REGISTRY['status']=='chained_to_2050').sum()} chained, "
      f"{(REGISTRY['status']=='retired_at_2050').sum()} retired)")


no_transition_2050 chained from no_transition_2035 (solve_result_num=0)
early_access_2050 chained from early_access_2035 (solve_result_num=0)
late_access_2050 chained from late_access_2035 (solve_result_num=0)
early_access_brazil_2050 chained from early_access_2035 (solve_result_num=0)

Vintage registry: 596 rows (342 chained, 254 retired)


## 1. Electricity generation — rule (a)

`f_max = f_min` for all technologies with positive `ELECTRICITY` output in `Layers_in_out.csv`.
This locks each cluster to its existing installed capacity (from AETN 2024 / Census data already encoded in the sufficiency base).
If `f_min = 0`, then `f_max = 0` — no new construction of any generator type.


In [3]:
# Source: Layers_in_out.csv -- techs with positive ELECTRICITY coefficient are generators
lio = pd.read_csv("../data/Layers_in_out.csv", sep=";")
lio.columns = [c.strip() for c in lio.columns]
tech_col = lio.columns[0]

# Layers_in_out.csv reference-equality check (same pattern already used in the demande.ipynb
# notebooks). This feeds directly into Technologies.csv, which enters an EnergyScope run.
# Efficiencies are frozen at 2025 values for both projection horizons, so the reference is the
# 2025 reality file.
LIO_REFERENCE_PATH = "../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv"
lio_check_local = pd.read_csv("../data/Layers_in_out.csv", sep=";", header=0, index_col=0)
lio_check_reference = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

problems = []
only_local_rows = sorted(set(lio_check_local.index) - set(lio_check_reference.index))
only_reference_rows = sorted(set(lio_check_reference.index) - set(lio_check_local.index))
if only_local_rows:
    problems.append(f"technologies only in local file: {only_local_rows}")
if only_reference_rows:
    problems.append(f"technologies only in EnergyScope reference: {only_reference_rows}")

only_local_cols = sorted(set(lio_check_local.columns) - set(lio_check_reference.columns))
only_reference_cols = sorted(set(lio_check_reference.columns) - set(lio_check_local.columns))
if only_local_cols:
    problems.append(f"layers only in local file: {only_local_cols}")
if only_reference_cols:
    problems.append(f"layers only in EnergyScope reference: {only_reference_cols}")

common_rows = sorted(set(lio_check_local.index) & set(lio_check_reference.index))
common_cols = sorted(set(lio_check_local.columns) & set(lio_check_reference.columns))
changed_rows = sorted(
    tech for tech in common_rows
    if not lio_check_local.loc[tech, common_cols].equals(lio_check_reference.loc[tech, common_cols])
)
if changed_rows:
    problems.append(f"technologies with different coefficients: {changed_rows}")

if problems:
    raise ValueError(
        f"Layers_in_out.csv differs from the EnergyScope reference ({LIO_REFERENCE_PATH}): "
        + "; ".join(problems)
    )
print(f"OK -- Layers_in_out.csv matches the EnergyScope reference ({LIO_REFERENCE_PATH})")

ELECTRICITY_GENERATORS = set(
    lio.loc[(lio["ELECTRICITY"] > 0) & (lio[tech_col] != "ELECTRICITY"), tech_col].tolist()
)
print(f"Identified {len(ELECTRICITY_GENERATORS)} electricity generators:")
print(sorted(ELECTRICITY_GENERATORS))


OK -- Layers_in_out.csv matches the EnergyScope reference (../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/00_INDEP/Layers_in_out.csv)


Identified 54 electricity generators:
['BFB_ST_BIOMASS', 'BIOMASS_TO_DIESEL', 'BIOMASS_TO_GASOLINE', 'BIOMASS_TO_JET_FUEL', 'BIOMASS_TO_LFO', 'BIOMASS_TO_METHANOL', 'BIOMASS_TO_POWER', 'BIOWASTE_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_JET_FUEL', 'BIOWASTE_TO_LFO', 'BIOWASTE_TO_METHANOL', 'BIO_HYDROLYSIS', 'CCGT', 'CCGT_AMMONIA', 'CCGT_SUR', 'CFB_ST_BIOMASS', 'COAL_IGCC', 'COAL_US', 'DEC_ADVCOGEN_GAS', 'DEC_ADVCOGEN_H2', 'DEC_COGEN_GAS', 'DEC_COGEN_OIL', 'DHN_COGEN_GAS', 'DHN_COGEN_WASTE', 'DHN_COGEN_WOOD', 'ETHANOL_TO_FUELS', 'FB_ST_BIOMASS', 'FUEL_CELL', 'GENSET_DIESEL', 'GEOTHERMAL', 'HS_DIESEL', 'HYDRO_DAM', 'HYDRO_RIVER', 'IND_COGEN_GAS', 'IND_COGEN_WASTE', 'IND_COGEN_WOOD', 'NUCLEAR', 'NUCLEAR_SMR', 'OCGT', 'PT_POWER_BLOCK', 'PV_HS', 'PV_ROOFTOP', 'PV_UTILITY', 'PYROLYSIS_TO_FUELS', 'PYROLYSIS_TO_LFO', 'ST_BIOMASS', 'ST_POWER_BLOCK', 'ST_SNG', 'TIDAL_RANGE', 'TIDAL_STREAM', 'WAVE', 'WIND_OFFSHORE', 'WIND_ONSHORE']


## 2. Off-grid and storage — rules (b) & (c)

**Rule (b):** `PV_HS` and `HS_DIESEL` → `f_min = 0` only. The 2012-vintage legacy kits are past their
catalogue lifetime (PV_HS 20y, HS_DIESEL 5y) by this horizon, so `f_min = 0`, but `f_max` is left
unconstrained (inherited from the sufficiency base, `1e15`) so the model can invest in new SHS
capacity.

**Rule (c):** All other battery/storage technologies → `f_max = f_min` (no new investment). `BATT_HS`
is deliberately excluded from `STORAGE_TECHS` — left untouched, it keeps the sufficiency base's own
values (`f_min = 0`, `f_max = 1e15`; its catalogue lifetime is 10y and it is likewise past
end-of-life). Excluding it matters because otherwise rule (c) would lock it to `f_max = f_min = 0`,
leaving no buildable battery for the PV_HS/BATT_HS coupling constraint (`.mod` default `= 1`) to pair
against new `PV_HS` capacity.


In [4]:
# Rule (b): off-grid techs disabled (f_min only)
OFF_GRID_TECHS = ["PV_HS", "HS_DIESEL"]

# Rule (c): storage techs locked (f_max = f_min)
# Includes electrical, thermal, chemical and vehicle storage.
# BATT_HS is deliberately excluded -- see rule (b)/(c) markdown above.
STORAGE_TECHS = [
    # Electrical / electrochemical
    "BATT_LI", "CAES", "BEV_BATT", "PHEV_BATT", "DAM_STORAGE", "PHS",
    # Thermal
    "TS_DEC_DIRECT_ELEC", "TS_DEC_HP_ELEC", "TS_DEC_THHP_GAS",
    "TS_DEC_COGEN_GAS",   "TS_DEC_COGEN_OIL", "TS_DEC_ADVCOGEN_GAS",
    "TS_DEC_ADVCOGEN_H2", "TS_DEC_BOILER_GAS", "TS_DEC_BOILER_WOOD",
    "TS_DEC_BOILER_OIL",  "TS_DHN_DAILY", "TS_DHN_SEASONAL", "TS_HIGH_TEMP", "TS_COLD",
    # Chemical / other
    "GAS_STORAGE", "H2_STORAGE", "CO2_STORAGE", "AMMONIA_STORAGE",
    "METHANOL_STORAGE", "PT_STORAGE", "ST_STORAGE",
]
print("OFF_GRID_TECHS:", OFF_GRID_TECHS)
print(f"STORAGE_TECHS ({len(STORAGE_TECHS)} entries) defined.")


OFF_GRID_TECHS: ['PV_HS', 'HS_DIESEL']
STORAGE_TECHS (27 entries) defined.


## 3. Cooking stoves — rule (d)

`f_min` for `STOVE_WOOD` and `STOVE_LPG` is derived from Census 2024 cooking fuel data
(file: `CSV_final_in_excel.xlsx`, sheet `data`, rows start at row 4):

| Column (0-indexed) | Field |
|---|---|
| 47 | Leña (wood households) |
| 50 | Gas domiciliario por cañería |
| 51 | Gas en garrafa |

$$\text{wood\_hh} = \text{col}_{47}, \quad \text{lpg\_hh} = \text{col}_{50} + \text{col}_{51}$$

$$f_{\min}^{\text{WOOD}} = \frac{\text{wood\_hh} \times 0.001344023}{0.1875 \times 8760} \;[\text{GW}]$$

$$f_{\min}^{\text{LPG}} = \frac{\text{lpg\_hh} \times 0.001344023}{0.1875 \times 8760} \;[\text{GW}]$$

Where $0.001344023$ GWh/hh/yr is the Census-based useful cooking energy intensity,
and $0.1875$ is the stove capacity factor (`c_p`).

**Disambiguation:** Two municipalities are named *Santa Rosa*.
Department Beni → `Santa_Rosa_Beni` → C1; Department Pando → `Santa_Rosa_Pando` → C4.


In [5]:
# Source: Bolivia Census 2024 -- CSV_final_in_excel.xlsx
xl = pd.ExcelFile("../../exctraction of data/output/CSV_final_in_excel.xlsx")
raw = xl.parse(xl.sheet_names[0], header=None)
data = raw.iloc[3:].reset_index(drop=True)  # skip 3 header rows

COOKING_HH = {}  # muni_key -> (wood_hh, lpg_hh)
for _, row in data.iterrows():
    muni = str(row[3]).strip() if pd.notna(row[3]) else ""
    dept = str(row[1]).strip() if pd.notna(row[1]) else ""
    if not muni or muni == "nan":
        continue
    wood_hh = int(row[47]) if pd.notna(row[47]) else 0
    gas_dom  = int(row[50]) if pd.notna(row[50]) else 0
    gas_gar  = int(row[51]) if pd.notna(row[51]) else 0
    lpg_hh   = gas_dom + gas_gar
    # Disambiguate the two "Santa Rosa" entries
    if muni == "Santa Rosa" and dept == "Beni":
        key = "Santa_Rosa_Beni"
    elif muni == "Santa Rosa" and dept == "Pando":
        key = "Santa_Rosa_Pando"
    else:
        key = muni.replace(" ", "_")
    COOKING_HH[key] = (wood_hh, lpg_hh)

COOK_INTENSITY = 0.001344023  # GWh per household per year (Census 2024 cooking energy intensity)
CP_STOVE       = 0.1875       # capacity factor for all stoves

stove_wood_fmin = {}
stove_lpg_fmin  = {}
for k, munis in CLUSTERS.items():
    wood_total = sum(COOKING_HH.get(m, (0, 0))[0] for m in munis)
    lpg_total  = sum(COOKING_HH.get(m, (0, 0))[1] for m in munis)
    stove_wood_fmin[k] = (wood_total * COOK_INTENSITY) / (CP_STOVE * 8760)
    stove_lpg_fmin[k]  = (lpg_total  * COOK_INTENSITY) / (CP_STOVE * 8760)

print(f"{'':8} {'wood_hh':>9} {'lpg_hh':>9} {'STOVE_WOOD f_min (GW)':>22} {'STOVE_LPG f_min (GW)':>22}")
for k, munis in CLUSTERS.items():
    wood_total = sum(COOKING_HH.get(m, (0, 0))[0] for m in munis)
    lpg_total  = sum(COOKING_HH.get(m, (0, 0))[1] for m in munis)
    print(f"C{k}       {wood_total:>9} {lpg_total:>9} {stove_wood_fmin[k]:>22.7f} {stove_lpg_fmin[k]:>22.7f}")


           wood_hh    lpg_hh  STOVE_WOOD f_min (GW)   STOVE_LPG f_min (GW)
C1            5017      5656              0.0041053              0.0046282
C2             340       452              0.0002782              0.0003699
C3            7014     32377              0.0057394              0.0264934
C4            5775     10520              0.0047256              0.0086083
C5             300     14696              0.0002455              0.0120254


## 4. Assemble capacity overrides (rules a–e), per scenario

Builds the capacity-only dataframe per **(scenario, cluster)** pair (8 columns, same shape as the
sufficiency base) -- scenario-aware for the first time, because rule (e) below now differs by
scenario (each chains from its own 2035 solve). The cost/lifetime block is merged in separately in
Section 5 (rule f) -- not saved yet.

**Correction (2026-08-05, found before any 2050 solve was attempted):** rule (a)'s generic
`f_max = f_min` lock, applied to rule (e)'s chained `f_min`, silently zeroed out
`GENSET_DIESEL.f_max` in `no_transition` C3/C4/C5 (its chained `f_min` is 0 everywhere -- it was
never unlocked past its legacy floor at 2035 in any scenario, see decision-record markdown above).
Combined with `PV_UTILITY`/`BATT_LI` also locked at their (zero, in `no_transition`) chained
`f_min`, those three clusters would have had **zero buildable electricity generation technology at
all** -- the run would be infeasible or physically meaningless, not merely pessimistic. "No
transition" means the existing thermal fleet renews itself at its own expense, not that it
disappears: rule (e)'s `f_min = 0` correctly removes the free brownfield floor (no legacy capacity
survives to 2050), but must not also remove the *technology* itself.

**Correction extended (2026-08-05b, after diagnosing a C4 `PV_UTILITY` overbuild-and-curtail
pattern in `early_access`):** the same `f_max = f_min = 0` artefact -- not a scenario decision --
also hit `GENSET_DIESEL` in `late_access`, `early_access`, and `early_access_brazil` at 2050, since
`GENSET_DIESEL` was never unlocked past its legacy floor at 2035 in *any* scenario (confirmed:
`F[2035,c]` exactly equals the legacy floor everywhere, so the chained `f_min[2050,c] = 0`
uniformly). `f_max = 0` on a technology whose floor merely aged to zero reads as "building a diesel
genset is forbidden," which is not what "the 2025 fleet is no longer free after 30 years" means.
Diagnosed via C4 in `early_access`: with `GENSET_DIESEL` fully unavailable and inter-cluster
transfer capacity left under-invested (well below its `tc_max` ceiling), `PV_UTILITY` was the only
flexible source left to satisfy the hourly demand-balance equality, forcing capacity to nearly
self-sufficiently cover the year's harder solar typical days (byte-identical `Time_series.csv`
between 2035/2050 ruled out a resource-data cause) and curtailing roughly half its own potential
production on sunnier days -- since `Curt` is costless (`.mod:249,378`), a cost-minimizing LP would
never build capacity *for* curtailment; the only way to observe this pattern is a binding
constraint forcing the capacity, which is exactly what a stray `f_max = 0` on the only remaining
dispatchable option does. Rule (i) below therefore now reopens `GENSET_DIESEL.f_max` in **every**
2050 scenario, immediately after rule (a) locks it, restoring the technology as a paid option
without reinstating any free capacity (`f_min` stays 0 everywhere). `PV_UTILITY`/`BATT_LI` are
unchanged by this rule -- `no_transition`'s stay locked at their chained `f_min` (intentional: no
supply-side transition = no new renewable investment), the other three scenarios' stay open as
already unlocked at solve time by `scripts/run.py`. **2035 is not touched**: there, the legacy fleet
still exists and its own `f_max = f_min` lock is correct (this 2050 pattern is only possible once a
floor has aged to exactly zero, which cannot happen at 2035 -- the 2035 lock is analogous to the
2025 `reality_phase2` scenario, where the existing fleet stays installed with no new construction).

A broader sweep (every `ELECTRICITY_GENERATORS` technology with nonzero 2035 `f_min` but deployed
`f_max[2050] = 0`, across all 4 scenarios, re-run after this extension) confirms only `GENSET_DIESEL`
(now fixed everywhere) and `PV_UTILITY` in `no_transition` (intentionally locked, not a bug --
`no_transition` builds no new solar by definition) show this pattern -- no other technology is
affected by the same mechanism.


In [6]:
capacity_only = {}  # scenario -> {k -> df}
for scenario in DEPLOY_SCENARIOS_2050:
    capacity_only[scenario] = {}
    for k in range(1, 6):
        df = suff[k].copy()

        # Rule (e): vintage-aware capacity chaining from 2035 -- overrides f_min for every
        # technology except EFFICIENCY (see CHAINED_FMIN_2050 above). Applied before Rule (a)
        # locks f_max = f_min, same position in the pipeline as the old EXISTING_FLEET_FMIN_GW
        # rule it replaces.
        chained = CHAINED_FMIN_2050[scenario][k]
        mask_chain = df["Technologies param"] != "EFFICIENCY"
        df.loc[mask_chain, "f_min"] = df.loc[mask_chain, "Technologies param"].map(chained)

        # Rule (g): DEC_SOLAR.f_max inherited from the deployed reality_access reference, not the
        # stale sufficiency base (see DEC_SOLAR_FMAX_BY_CLUSTER above) -- not touched by any other
        # rule below (not a generator, not off-grid, not storage), so this is a plain override.
        df.loc[df["Technologies param"] == "DEC_SOLAR", "f_max"] = DEC_SOLAR_FMAX_BY_CLUSTER[k]

        # Rule (a): Lock all electricity generation -- f_max = f_min
        # OFF_GRID_TECHS excluded: Layers_in_out.csv flags PV_HS/HS_DIESEL as ELECTRICITY
        # generators, so without this exclusion Rule (a) would re-lock f_max = f_min = 0 for them
        # here, undoing rule (b) before it even runs.
        mask_gen = df["Technologies param"].isin(ELECTRICITY_GENERATORS - set(OFF_GRID_TECHS))
        df.loc[mask_gen, "f_max"] = df.loc[mask_gen, "f_min"]

        # Rule (i) (correction, extended 2026-08-05b, see markdown above): GENSET_DIESEL -- reopen
        # f_max after rule (a) locked it to the chained f_min (0 in every cluster, every scenario --
        # it was never unlocked past its legacy floor at 2035 anywhere). f_max=0 on a floor that
        # merely aged to zero reads as "forbidden to build", not "no longer free" -- applies to
        # EVERY 2050 scenario now (originally no_transition only; extended after the same artefact
        # was found forcing a PV_UTILITY overbuild-and-curtail pattern in early_access/C4).
        # f_min stays 0 everywhere (no free brownfield floor). PV_UTILITY/BATT_LI are untouched by
        # this rule: no_transition's stay locked at their chained f_min (intentional -- no
        # supply-side transition), the others stay open as already unlocked at solve time by
        # scripts/run.py. 2035 is not touched by this rule at all (not in this notebook's scope).
        mask_gd = df["Technologies param"] == "GENSET_DIESEL"
        df.loc[mask_gd, "f_max"] = 1e15

        # Rule (b): Disable off-grid legacy kits -- f_min = 0 only (f_max is left unconstrained,
        # inherited from sufficiency, so the model can build new kits)
        for tech in OFF_GRID_TECHS:
            mask = df["Technologies param"] == tech
            df.loc[mask, "f_min"] = 0.0

        # Rule (h): PV_HS / HS_DIESEL f_max_prod capped at the cluster's dispersed demand (see
        # DISPERSED_DEMAND_GWH_BY_CLUSTER above). f_min_prod left at 0 (its default) -- not pinned.
        for tech in OFF_GRID_TECHS:
            mask = df["Technologies param"] == tech
            df.loc[mask, "f_max_prod"] = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
            df.loc[mask, "f_min_prod"] = 0.0

        # Rule (c): Lock storage -- f_max = f_min (no new storage investment); BATT_HS excluded
        mask_stor = df["Technologies param"].isin(STORAGE_TECHS)
        df.loc[mask_stor, "f_max"] = df.loc[mask_stor, "f_min"]

        # Disable ST_SNG: f_min = f_max = 0 for all clusters
        df.loc[df["Technologies param"] == "ST_SNG", "f_min"] = 0.0
        df.loc[df["Technologies param"] == "ST_SNG", "f_max"] = 0.0

        # LED-only lighting: conventional bulb/tube techs disabled (f_max = 0)
        df.loc[df["Technologies param"] == "CONVENTIONAL_BULB",  "f_max"] = 0.0
        df.loc[df["Technologies param"] == "CONVENTIONAL_LIGHT", "f_max"] = 0.0

        # Rule (d): Stove f_min from Census 2024 -- f_max unchanged from sufficiency. Always wins
        # for STOVE_WOOD/STOVE_LPG regardless of rule (e)'s non-survivor classification for them
        # (frozen, un-projected cooking-demand requirement, independent of vintage carryover).
        df.loc[df["Technologies param"] == "STOVE_WOOD", "f_min"] = stove_wood_fmin[k]
        df.loc[df["Technologies param"] == "STOVE_LPG",  "f_min"] = stove_lpg_fmin[k]

        capacity_only[scenario][k] = df

print(f"Computed capacity overrides (rules a-e) for {len(DEPLOY_SCENARIOS_2050)} scenarios x C1-C5")


Computed capacity overrides (rules a-e) for 4 scenarios x C1-C5


## 5. Cost/lifetime block — rule (f)

`c_inv`, `c_maint`, `gwp_constr`, `lifetime` (plus the structural `Category`/`Subcategory`/
`Technologies name`/`Comment` columns) are merged in verbatim from `COST_REF_PATH`
(`Data/2025/reality/02_REF_REGION/Technologies.csv`), the deployed and calibrated Norte Amazónica
2025 catalog -- **not** projected or trended in any way for this horizon, and identical across all
4 scenarios (cost trajectories don't depend on the demand-side scenario). Capacity columns
(`c_p`, `fmin_perc`, `fmax_perc`, `f_min`, `f_max`, `f_min_prod`, `f_max_prod`) are exactly the
`capacity_only` dataframes computed in Section 4 (rules a–e), scenario-aware, and are untouched by
this merge.

A blocking assertion immediately below checks that `c_inv`/`c_maint`/`gwp_constr`/`lifetime` are
identical to the 2025 reference for every technology in every scenario/cluster, with **no
exceptions** -- cost trajectories for 2050 are not modelled yet. If the assertion fails, the
notebook raises before writing any `Technologies.csv`, so a partially-wrong catalogue is never
deployed.


In [7]:
# Rule (f): merge cost/lifetime block from the calibrated 2025 reality catalog.
COST_COLS = ["Category", "Subcategory", "Technologies name", "c_inv", "c_maint", "gwp_constr",
             "lifetime", "Comment"]
DEPLOYED_COLUMNS = ["Category", "Subcategory", "Technologies name", "Technologies param",
                    "c_inv", "c_maint", "gwp_constr", "lifetime",
                    "c_p", "fmin_perc", "fmax_perc", "f_min", "f_max",
                    "f_min_prod", "f_max_prod", "Comment"]

# skiprows=[1]: the deployed file has a units-description row right after the header
cost_ref = pd.read_csv(COST_REF_PATH, sep=";", skiprows=[1])
cost_ref["Technologies param"] = cost_ref["Technologies param"].str.strip()
cost_ref = cost_ref.set_index("Technologies param")

assembled = {}  # scenario -> {k -> df}
for scenario, by_cluster in capacity_only.items():
    assembled[scenario] = {}
    for k, df in by_cluster.items():
        df = df.set_index("Technologies param")
        missing = set(df.index) - set(cost_ref.index)
        if missing:
            raise ValueError(f"{scenario} C{k}: technologies missing from the 2025 cost reference: {sorted(missing)}")
        df[COST_COLS] = cost_ref.loc[df.index, COST_COLS]

        # NREL ATB Moderate/Mid trajectory -- ratio applied to c_inv only, never an absolute
        # substitution. c_maint, gwp_constr, lifetime are left untouched (verbatim from cost_ref).
        for tech, ratio_by_year in NREL_ATB_C_INV_RATIO.items():
            if tech in df.index:
                df.loc[tech, "c_inv"] = df.loc[tech, "c_inv"] * ratio_by_year[2050]

        assembled[scenario][k] = df.reset_index()[DEPLOYED_COLUMNS]

print(f"Merged cost/lifetime block from {COST_REF_PATH} for {len(DEPLOY_SCENARIOS_2050)} scenarios x C1-C5 "
      f"({len(cost_ref)} technologies)")


Merged cost/lifetime block from ../../../EnergyScope_BO_nord_amazonia/Data/2025/reality/02_REF_REGION/Technologies.csv for 4 scenarios x C1-C5 (272 technologies)


In [8]:
# Blocking assertion -- c_maint, gwp_constr, lifetime must be IDENTICAL to the 2025 reality
# reference for every technology, in every scenario, with NO exceptions. c_inv must be IDENTICAL
# to the 2025 reference for every technology EXCEPT the four NREL ATB technologies (PV_UTILITY,
# BATT_LI, PV_HS, BATT_HS), whose c_inv must instead equal exactly ratio * 2025 c_inv
# (NREL_ATB_C_INV_RATIO, Moderate/Mid trajectory). Raises and halts before any file is written.
COST_ASSERT_COLS = ["c_maint", "gwp_constr", "lifetime"]
ref_check = cost_ref[COST_ASSERT_COLS]
ratio_techs = set(NREL_ATB_C_INV_RATIO)

for scenario, by_cluster in assembled.items():
    for k, df in by_cluster.items():
        check = df.set_index("Technologies param")[COST_ASSERT_COLS]
        diff = (check - ref_check.loc[check.index]).abs() > 1e-9
        bad = diff.any(axis=1)
        if bad.any():
            raise AssertionError(
                f"{scenario} C{k}: cost/lifetime columns differ from the 2025 reality reference for "
                f"{check.index[bad].tolist()} -- no exceptions are allowed at this horizon "
                f"(cost trajectories not yet applied to c_maint/gwp_constr/lifetime)."
            )

        c_inv_check = df.set_index("Technologies param")["c_inv"]
        ref_c_inv = cost_ref["c_inv"]
        flat_techs = [t for t in c_inv_check.index if t not in ratio_techs]
        diff_flat = (c_inv_check.loc[flat_techs] - ref_c_inv.loc[flat_techs]).abs() > 1e-9
        if diff_flat.any():
            raise AssertionError(
                f"{scenario} C{k}: c_inv differs from the 2025 reality reference for "
                f"{c_inv_check.loc[flat_techs].index[diff_flat].tolist()} -- no exceptions are "
                f"allowed outside NREL_ATB_C_INV_RATIO."
            )
        for tech in ratio_techs:
            actual_ratio = c_inv_check[tech] / ref_c_inv[tech]
            expected_ratio = NREL_ATB_C_INV_RATIO[tech][2050]
            if abs(actual_ratio - expected_ratio) > 1e-9:
                raise AssertionError(
                    f"{scenario} C{k}: {tech} c_inv/2025 ratio = {actual_ratio}, expected exactly "
                    f"{expected_ratio} (NREL_ATB_C_INV_RATIO, source: NREL ATB 2024 Moderate/Mid)."
                )

print(f"ASSERT OK -- c_maint, gwp_constr, lifetime identical to 2025 reality for all "
      f"{len(cost_ref)} technologies, {len(DEPLOY_SCENARIOS_2050)} scenarios x C1-C5 (no exceptions); "
      f"c_inv identical to 2025 reality for all but {sorted(ratio_techs)}, which match the "
      f"announced NREL ATB ratio exactly")


ASSERT OK -- c_maint, gwp_constr, lifetime identical to 2025 reality for all 272 technologies, 4 scenarios x C1-C5 (no exceptions); c_inv identical to 2025 reality for all but ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], which match the announced NREL ATB ratio exactly


## 6. Save `Technologies.csv` (local build output)

In [9]:
for scenario, by_cluster in assembled.items():
    for k, df in by_cluster.items():
        out_path = os.path.join(OUT_DIR_BY_SCENARIO[scenario], f"C{k}", "Technologies.csv")
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        df.to_csv(out_path, sep=";", index=False)

print(f"Saved Technologies.csv for {len(DEPLOY_SCENARIOS_2050)} scenarios x C1-C5 (full 16-column deployed format)")


Saved Technologies.csv for 4 scenarios x C1-C5 (full 16-column deployed format)


## 7. Verification

In [10]:
clusters_out = {}  # scenario -> {k -> df}
for scenario in DEPLOY_SCENARIOS_2050:
    clusters_out[scenario] = {}
    for k in range(1, 6):
        path = os.path.join(OUT_DIR_BY_SCENARIO[scenario], f"C{k}", "Technologies.csv")
        df = pd.read_csv(path, sep=";")
        df["Technologies param"] = df["Technologies param"].str.strip()
        clusters_out[scenario][k] = df

def lookup(df, tech, col):
    row = df.loc[df["Technologies param"] == tech, col]
    return float(row.values[0]) if len(row) else float("nan")

header = f"{'Technology':<32}" + "".join(f"  C{k:>11}" for k in range(1, 6))
sep    = "-" * len(header)

# --- Rule (e): vintage-aware chaining, cross-check against CHAINED_FMIN_2050 (cell 0bis/3) ---
# Not hardcoded: rule (e) reads solved 2035 F values, which shift whenever 2035 is resolved
# under different costs (e.g. after applying NREL_ATB_C_INV_RATIO). This check must always
# compare against the live CHAINED_FMIN_2050 computed in this same run, not a frozen snapshot.
print("=== Rule (e): vintage-aware chaining -- deployed f_min vs CHAINED_FMIN_2050 (live) ===")
print(f"{'scenario':<14}{'cluster':<9}{'tech':<16}{'deployed':>12}{'expected':>12}  ok?")
SPECIAL_LEGACY_CHECK = [("GENSET_DIESEL", k) for k in (3, 4, 5)] + [("PV_UTILITY", k) for k in (4, 5)]
for scenario in DEPLOY_SCENARIOS_2050:
    for tech, k in SPECIAL_LEGACY_CHECK:
        actual = lookup(clusters_out[scenario][k], tech, "f_min")
        expected = CHAINED_FMIN_2050[scenario][k][tech]
        ok = abs(actual - expected) < 5e-6
        assert ok, f"{scenario} C{k} {tech} f_min = {actual}, expected {expected} (CHAINED_FMIN_2050)"
        print(f"{scenario:<14}C{k:<8}{tech:<16}{actual:>12.6f}{expected:>12.6f}  OK")
print("ASSERT OK -- all deployed f_min values match CHAINED_FMIN_2050 exactly")
print()

# --- Rule (a)+(e): generators locked (f_max == f_min) in every scenario -- except GENSET_DIESEL,
# reopened everywhere by rule (i) (checked separately right below) ---
print("=== Rule (a)+(e): generators locked (f_max == f_min) in every scenario/cluster ===")
for scenario in DEPLOY_SCENARIOS_2050:
    for tech in ["PV_UTILITY"]:  # GENSET_DIESEL is reopened everywhere by rule (i), checked below
        for k in range(1, 6):
            fmin = lookup(clusters_out[scenario][k], tech, "f_min")
            fmax = lookup(clusters_out[scenario][k], tech, "f_max")
            assert abs(fmax - fmin) < 1e-9, f"{scenario} C{k} {tech}: f_max={fmax} != f_min={fmin}"
print("ASSERT OK -- PV_UTILITY locked (f_max == f_min) in every scenario/cluster "
      "(GENSET_DIESEL checked separately below -- reopened everywhere by rule (i))")
print()

# --- Rule (i) correction (extended 2026-08-05b): GENSET_DIESEL reopened (f_min=0, f_max open) in
# EVERY 2050 scenario, since it was never unlocked past its legacy floor at 2035 anywhere ---
print("=== Rule (i): GENSET_DIESEL reopened (f_min=0, f_max=Infinity) in every 2050 scenario ===")
for scenario in DEPLOY_SCENARIOS_2050:
    vals_fmin = [lookup(clusters_out[scenario][k], "GENSET_DIESEL", "f_min") for k in range(1, 6)]
    vals_fmax = [lookup(clusters_out[scenario][k], "GENSET_DIESEL", "f_max") for k in range(1, 6)]
    print(f"{scenario:<22}f_min" + "".join(f"  {v:>11.5f}" for v in vals_fmin))
    print(f"{'':<22}f_max" + "".join(f"  {v:>11.1e}" for v in vals_fmax))
    for k in range(1, 6):
        fmin = lookup(clusters_out[scenario][k], "GENSET_DIESEL", "f_min")
        fmax = lookup(clusters_out[scenario][k], "GENSET_DIESEL", "f_max")
        assert fmin == 0.0, f"{scenario} C{k} GENSET_DIESEL f_min={fmin}, expected 0.0 (no free brownfield floor)"
        assert fmax >= 1e14, f"{scenario} C{k} GENSET_DIESEL f_max={fmax}, expected open (>=1e14)"
print("ASSERT OK -- GENSET_DIESEL: f_min=0 (no free floor), f_max open in every cluster, "
      "all 4 scenarios")
print()

# --- Rule (b): legacy off-grid kits dead (f_min = 0), but buildable (f_max > 0) ---
print("=== Rule (b): legacy kits dead (f_min = 0), new kits buildable (f_max > 0) -- early_access shown ===")
print(header); print(sep)
for tech in ["PV_HS", "HS_DIESEL", "BATT_HS"]:
    for col in ["f_min", "f_max"]:
        vals = [lookup(clusters_out["early_access"][k], tech, col) for k in range(1, 6)]
        print(f"{tech+' '+col:<32}" + "".join(f"  {v:>11.5f}" for v in vals))
for scenario in DEPLOY_SCENARIOS_2050:
    for tech in ["PV_HS", "HS_DIESEL"]:
        for k in range(1, 6):
            fmin = lookup(clusters_out[scenario][k], tech, "f_min")
            fmax = lookup(clusters_out[scenario][k], tech, "f_max")
            assert fmin == 0.0, f"{scenario} C{k} {tech} f_min = {fmin}, expected 0.0"
            assert fmax > 0.0, f"{scenario} C{k} {tech} f_max = {fmax}, expected > 0"
print("ASSERT OK -- PV_HS/HS_DIESEL f_min=0 (legacy dead), f_max>0 (new kits buildable) in every scenario/cluster")
print()

# --- LED-only lighting ---
print("=== LED-only lighting (CONVENTIONAL_BULB / CONVENTIONAL_LIGHT f_max == 0) -- early_access shown ===")
for tech in ["CONVENTIONAL_BULB", "CONVENTIONAL_LIGHT"]:
    for scenario in DEPLOY_SCENARIOS_2050:
        for k in range(1, 6):
            fmax = lookup(clusters_out[scenario][k], tech, "f_max")
            assert fmax == 0.0, f"{scenario} C{k} {tech} f_max = {fmax}, expected 0.0"
print("ASSERT OK -- CONVENTIONAL_BULB/CONVENTIONAL_LIGHT f_max=0 in every scenario/cluster")
print()

# --- Rule (e) non-survivors: f_min = 0 regardless of scenario (except stoves, see rule d note) ---
print("=== Rule (e): non-survivor technologies -- f_min = 0 in every scenario/cluster ===")
NON_SURVIVOR_CHECK = [t for t in NON_SURVIVOR_TECHS_2050 if t not in ("STOVE_WOOD", "STOVE_LPG")]
for tech in NON_SURVIVOR_CHECK:
    for scenario in DEPLOY_SCENARIOS_2050:
        for k in range(1, 6):
            fmin = lookup(clusters_out[scenario][k], tech, "f_min")
            assert fmin == 0.0, f"{scenario} C{k} {tech} f_min = {fmin}, expected 0.0 (non-survivor)"
print(f"ASSERT OK -- {NON_SURVIVOR_CHECK} all f_min=0 in every scenario/cluster")
print("(STOVE_WOOD/STOVE_LPG excluded from this check -- rule (d) always overrides them afterward,")
print(" see Section 3/4; their non-survivor classification under rule (e) is moot by construction.)")
print()

# --- Rule (d): stove f_min (frozen at Census 2024 -- not projected to this horizon) ---
print("=== Rule (d): stove f_min (Census 2024, NOT projected) -- identical across scenarios ===")
print(header); print(sep)
for tech in ["STOVE_WOOD", "STOVE_LPG"]:
    vals = [lookup(clusters_out["no_transition"][k], tech, "f_min") for k in range(1, 6)]
    print(f"{tech+' f_min':<32}" + "".join(f"  {v:>11.7f}" for v in vals))
    for scenario in DEPLOY_SCENARIOS_2050:
        for k in range(1, 6):
            v = lookup(clusters_out[scenario][k], tech, "f_min")
            expected = stove_wood_fmin[k] if tech == "STOVE_WOOD" else stove_lpg_fmin[k]
            assert abs(v - expected) < 1e-9, f"{scenario} C{k} {tech} f_min={v}, expected {expected}"
print("ASSERT OK -- STOVE_WOOD/STOVE_LPG f_min matches Census 2024 calc in every scenario/cluster")
print()

print("=== Efficiencies frozen: Layers_in_out.csv verified identical to the 2025 reality reference (Section 1) ===")
print()
print("Cooking-demand-vs-stove-capacity check SKIPPED: it requires Demands.csv, which this notebook "
      "deliberately does not generate (Source A/B/C breakdown not yet available for this horizon -- "
      "see 'Missing entries' note at the end of this notebook).")
print()

# --- Rule (f): cost/lifetime block spot-check ---
print("=== Rule (f): cost/lifetime spot-check against the 2025 reality reference (C1, early_access shown) ===")
spot_checks = [
    ("GRID",       "c_inv",    0.0),
    ("HVAC_LINE",  "c_inv",    2.0),
    ("HVAC_LINE",  "c_maint",  0.04),
    ("PV_HS",      "c_inv",    2730.0 * NREL_ATB_C_INV_RATIO["PV_HS"][2050]),  # NREL ATB ratio applied
    ("PV_HS",      "c_maint",  10.0),
    ("PV_HS",      "lifetime", 20.0),
    ("HS_DIESEL",  "lifetime", 5.0),
    ("BATT_LI",    "lifetime", 15.0),
]
for tech, col, expected in spot_checks:
    actual = lookup(clusters_out["early_access"][1], tech, col)
    ok = "OK" if abs(actual - expected) < 1e-9 else "FAIL"
    print(f"  {tech:<12} {col:<10} = {actual:>12.5f}  (expected {expected:>10.5f})  [{ok}]")
print()

print("=== Rule (g): DEC_SOLAR.f_max spot-check against the deployed reality_access reference ===")
for scenario in DEPLOY_SCENARIOS_2050:
    for k in range(1, 6):
        actual = lookup(clusters_out[scenario][k], "DEC_SOLAR", "f_max")
        expected = DEC_SOLAR_FMAX_BY_CLUSTER[k]
        assert abs(actual - expected) < 1e-6, (
            f"{scenario} C{k} DEC_SOLAR f_max = {actual}, expected {expected}")
        assert actual > 0.0, f"{scenario} C{k} DEC_SOLAR f_max = {actual}, expected > 0"
print("ASSERT OK -- DEC_SOLAR.f_max matches the deployed reality_access reference and is nonzero everywhere")
print()

print("=== Rule (h): PV_HS / HS_DIESEL f_max_prod spot-check against cluster_summary.csv ===")
for scenario in DEPLOY_SCENARIOS_2050:
    for tech in ["PV_HS", "HS_DIESEL"]:
        for k in range(1, 6):
            actual = lookup(clusters_out[scenario][k], tech, "f_max_prod")
            expected = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
            assert abs(actual - expected) < 1e-6, (
                f"{scenario} C{k} {tech} f_max_prod = {actual}, expected {expected}")
            fminp = lookup(clusters_out[scenario][k], tech, "f_min_prod")
            assert fminp == 0.0, f"{scenario} C{k} {tech} f_min_prod = {fminp}, expected 0.0"
print("ASSERT OK -- PV_HS/HS_DIESEL f_max_prod matches cluster_summary.csv, f_min_prod stays 0, everywhere")


=== Rule (e): vintage-aware chaining -- deployed f_min vs CHAINED_FMIN_2050 (live) ===
scenario      cluster  tech                deployed    expected  ok?
no_transition C3       GENSET_DIESEL       0.000000    0.000000  OK
no_transition C4       GENSET_DIESEL       0.000000    0.000000  OK
no_transition C5       GENSET_DIESEL       0.000000    0.000000  OK
no_transition C4       PV_UTILITY          0.000000    0.000000  OK
no_transition C5       PV_UTILITY          0.000000    0.000000  OK
early_access  C3       GENSET_DIESEL       0.000000    0.000000  OK
early_access  C4       GENSET_DIESEL       0.000000    0.000000  OK
early_access  C5       GENSET_DIESEL       0.000000    0.000000  OK
early_access  C4       PV_UTILITY          0.059047    0.059047  OK
early_access  C5       PV_UTILITY          0.051503    0.051503  OK
late_access   C3       GENSET_DIESEL       0.000000    0.000000  OK
late_access   C4       GENSET_DIESEL       0.000000    0.000000  OK
late_access   C5       GENSE

ASSERT OK -- PV_HS/HS_DIESEL f_min=0 (legacy dead), f_max>0 (new kits buildable) in every scenario/cluster

=== LED-only lighting (CONVENTIONAL_BULB / CONVENTIONAL_LIGHT f_max == 0) -- early_access shown ===
ASSERT OK -- CONVENTIONAL_BULB/CONVENTIONAL_LIGHT f_max=0 in every scenario/cluster

=== Rule (e): non-survivor technologies -- f_min = 0 in every scenario/cluster ===
ASSERT OK -- ['BATT_LI', 'BATT_HS', 'HS_DIESEL', 'REFRIGERATOR_EL', 'DEC_DIRECT_ELEC', 'LED_BULB', 'LED_LIGHT', 'STOVE_NG', 'STOVE_OIL', 'STOVE_ELEC', 'COMM_MACHINERY_EL', 'FISH_MACHINERY_EL'] all f_min=0 in every scenario/cluster
(STOVE_WOOD/STOVE_LPG excluded from this check -- rule (d) always overrides them afterward,
 see Section 3/4; their non-survivor classification under rule (e) is moot by construction.)

=== Rule (d): stove f_min (Census 2024, NOT projected) -- identical across scenarios ===
Technology                        C          1  C          2  C          3  C          4  C          5
---------------

## Missing entries — `Demands.csv` not generated for this horizon

This notebook deliberately does **not** produce `output_energyscope_2050/{scenario}/C{k}/Demands.csv`.
Building it the way `demande.ipynb` does for 2025 needs three horizon-specific inputs that don't
exist yet:

1. **Source A (grid-connected demand)** — a 2050 equivalent of
   `exctraction of data/output/source_A_all_sectors_end_uses.csv`: AETN grid consumption by
   municipality/sector/end-use, projected to 2050. Only the 2024/2025 measured file exists today.
2. **Source B (off-grid RAMP demand)** — a 2050 equivalent of `data ramp/ramp_reality_annual_summary.csv`:
   a RAMP simulation run for the *projected* number of off-grid households per municipality at 2050
   (not the 2025 figure of 9,325 HH). The projected off-grid (dispersed) household count itself is
   already available at cluster level from the breakeven classification in
   `analyse_GIS_phase2_projections/output/2050/community_detail.csv`, but no RAMP run has been done
   yet against that projected household count, and no per-municipality (rather than per-cluster)
   breakdown exists.
3. **Cooking fuel mix** — a 2050 equivalent of `exctraction of data/output/CSV_final_in_excel.xlsx`:
   projected non-electric cooking households (wood/LPG) by municipality. Only the 2024 census split
   exists today; `STOVE_WOOD`/`STOVE_LPG` `f_min` in this notebook's `Technologies.csv` output is
   still computed from the unprojected 2024 census figures (Section 3 / rule (d)) — an implicit
   "frozen cooking behaviour" assumption, not requested for this horizon and not yet corrected.

`home_systems.ipynb` and `demande.ipynb` were copied into this folder (with their output path
suffixed to `output_energyscope_2050`) but were **not executed**, for the same reason: their 2025
inputs (2024 census equipment counts, `source_A_all_sectors_end_uses.csv`,
`ramp_reality_annual_summary.csv`) are not valid for 2050, and no projected replacement exists yet.
`home_systems.ipynb`'s output (legacy off-grid kit `f_min`) is moot regardless of that gap — those
values are forced to 0 directly in this notebook (rule (b)), since the 2012-vintage kits are past
their catalogue lifetime at this horizon.


## 8. Deploy `Layers_in_out.csv` to the case-study data directories

The project rule is frozen efficiencies: `Layers_in_out.csv` must be byte-identical to the 2025
reality reference (`{LIO_REFERENCE_PATH}`) at every horizon. Section 1 above already verifies the
local source file (`../data/Layers_in_out.csv`) matches it. Each deployed copy below is overwritten
with an exact copy of the 2025 reference and re-verified byte-for-byte plus value-for-value
(`DataFrame.equals`) — no partial or edited copies.


In [11]:
# Section 8: overwrite the deployed Layers_in_out.csv copies with an exact copy of the
# 2025 reality reference (frozen-efficiencies rule). Byte-exact copy + re-verification.
import filecmp
import shutil

DEPLOY_TARGETS_LIO_2050 = [
    f"../../../EnergyScope_BO_nord_amazonia/Data/2050/{scenario}/00_INDEP/Layers_in_out.csv"
    for scenario in DEPLOY_SCENARIOS_2050
]

lio_reference_check = pd.read_csv(LIO_REFERENCE_PATH, sep=";", header=0, index_col=0)

for target in DEPLOY_TARGETS_LIO_2050:
    shutil.copyfile(LIO_REFERENCE_PATH, target)

    if not filecmp.cmp(LIO_REFERENCE_PATH, target, shallow=False):
        raise AssertionError(f"{target} is not byte-identical to the 2025 reality reference after deployment")

    deployed_check = pd.read_csv(target, sep=";", header=0, index_col=0)
    if not deployed_check.equals(lio_reference_check):
        raise AssertionError(f"{target} does not match the 2025 reality reference after deployment")

    print(f"OK -- {target} deployed and verified byte-identical to the 2025 reality reference")


OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2050/no_transition/00_INDEP/Layers_in_out.csv deployed and verified byte-identical to the 2025 reality reference
OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2050/early_access/00_INDEP/Layers_in_out.csv deployed and verified byte-identical to the 2025 reality reference
OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2050/late_access/00_INDEP/Layers_in_out.csv deployed and verified byte-identical to the 2025 reality reference
OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2050/early_access_brazil/00_INDEP/Layers_in_out.csv deployed and verified byte-identical to the 2025 reality reference


## 9. Deploy `Technologies.csv` (capacity columns) to the case-study data directories

`Data/2050/{scenario}/C{k}/Technologies.csv` (the minimal 8-column capacity file actually read by
`esmc.utils.region.py:read_tech()` and merged onto the `02_REF_REGION` catalog at solve time) is
redeployed for all 5 clusters, in every 2050 scenario, from the just-recomputed **scenario-aware**
`assembled` dataframes (Section 5) -- this is the first build where the deployed `f_min` differs by
scenario for `PV_UTILITY`/`GENSET_DIESEL` and, in principle, for any other chained technology. No
manual CSV edits.


In [12]:
# Section 9: redeploy the 8-column capacity subset of Technologies.csv to every 2050
# case-study cluster directory, from the just-recomputed assembled dataframes (Section 5),
# scenario by scenario.
CAPACITY_COLS = ["Technologies param", "c_p", "fmin_perc", "fmax_perc",
                  "f_min", "f_max", "f_min_prod", "f_max_prod"]

for scenario in DEPLOY_SCENARIOS_2050:
    for k in range(1, 6):
        target = (f"../../../EnergyScope_BO_nord_amazonia/Data/2050/{scenario}/"
                  f"C{k}/Technologies.csv")
        assembled[scenario][k][CAPACITY_COLS].to_csv(target, sep=";", index=False)

        redeployed = pd.read_csv(target, sep=";", index_col=0)
        redeployed.index = redeployed.index.str.strip()
        actual = float(redeployed.loc["DEC_SOLAR", "f_max"])
        expected = DEC_SOLAR_FMAX_BY_CLUSTER[k]
        if abs(actual - expected) > 1e-6 or actual <= 0.0:
            raise AssertionError(
                f"{target}: DEC_SOLAR f_max = {actual}, expected {expected} (nonzero)")
        for tech in ["PV_HS", "HS_DIESEL"]:
            actual_fmp = float(redeployed.loc[tech, "f_max_prod"])
            expected_fmp = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
            if abs(actual_fmp - expected_fmp) > 1e-6:
                raise AssertionError(
                    f"{target}: {tech} f_max_prod = {actual_fmp}, expected {expected_fmp}")
            actual_fminp = float(redeployed.loc[tech, "f_min_prod"])
            if actual_fminp != 0.0:
                raise AssertionError(f"{target}: {tech} f_min_prod = {actual_fminp}, expected 0.0")
    print(f"OK -- Technologies.csv (C1-C5) redeployed for 2050/{scenario}, "
          f"DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster")


OK -- Technologies.csv (C1-C5) redeployed for 2050/no_transition, DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster


OK -- Technologies.csv (C1-C5) redeployed for 2050/early_access, DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster
OK -- Technologies.csv (C1-C5) redeployed for 2050/late_access, DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster
OK -- Technologies.csv (C1-C5) redeployed for 2050/early_access_brazil, DEC_SOLAR.f_max verified nonzero and matching reality_access in every cluster


## 9bis. Deploy cost/lifetime block (`c_inv` only changes) to `02_REF_REGION`

No prior step wrote the cost/lifetime block to `Data/2050/{scenario}/02_REF_REGION/Technologies.csv` -- it has so far been a static file, correctly matching the 2025 reality reference by construction (Section 5), never rewritten because nothing needed updating. Now that the NREL ATB ratio makes `c_inv` diverge from 2025 for four technologies, this step becomes necessary. Only the four cost/lifetime columns are touched; every other column (capacity, category, comment placeholder text) in the deployed file is preserved untouched, and the file's header + units row are preserved byte-for-byte.

In [13]:
COST_ONLY_COLS = ["c_inv", "c_maint", "gwp_constr", "lifetime"]

for scenario in DEPLOY_SCENARIOS_2050:
    target = (f"../../../EnergyScope_BO_nord_amazonia/Data/2050/{scenario}/"
              f"02_REF_REGION/Technologies.csv")

    with open(target, encoding="utf-8") as f:
        header_line = f.readline()
        units_line = f.readline()

    existing = pd.read_csv(target, sep=";", skiprows=[1])
    existing["Technologies param"] = existing["Technologies param"].str.strip()
    existing = existing.set_index("Technologies param")

    missing = set(cost_ref.index) - set(existing.index)
    if missing:
        raise ValueError(
            f"{target}: technologies missing from the deployed ref-region catalog: "
            f"{sorted(missing)}")

    # cost columns are cluster-independent (identical across C1-C5) -- any k works as source
    source_cost = assembled[scenario][1].set_index("Technologies param")[COST_ONLY_COLS]
    existing.loc[source_cost.index, COST_ONLY_COLS] = source_cost

    for tech in sorted(ratio_techs):
        ratio = NREL_ATB_C_INV_RATIO[tech][2050]
        note = (f"NREL ATB 2024 Moderate/Mid c_inv ratio 2050={ratio} "
                f"(source: analyse data ramp/2050/technologies.ipynb).")
        prior = existing.loc[tech, "Comment"]
        prior = "" if pd.isna(prior) else str(prior).strip()
        existing.loc[tech, "Comment"] = (prior + " " + note).strip() if prior else note

    with open(target, "w", encoding="utf-8", newline="") as f:
        f.write(header_line)
        f.write(units_line)
        existing.reset_index()[DEPLOYED_COLUMNS].to_csv(f, sep=";", index=False, header=False)

    # re-read and verify
    redeployed = pd.read_csv(target, sep=";", skiprows=[1])
    redeployed["Technologies param"] = redeployed["Technologies param"].str.strip()
    redeployed = redeployed.set_index("Technologies param")
    for tech in sorted(ratio_techs):
        actual_ratio = redeployed.loc[tech, "c_inv"] / cost_ref.loc[tech, "c_inv"]
        expected_ratio = NREL_ATB_C_INV_RATIO[tech][2050]
        if abs(actual_ratio - expected_ratio) > 1e-9:
            raise AssertionError(
                f"{target}: deployed {tech} c_inv/2025 ratio = {actual_ratio}, expected "
                f"exactly {expected_ratio}")
    other_techs = [t for t in redeployed.index if t not in ratio_techs]
    diff_other = (redeployed.loc[other_techs, "c_inv"] - cost_ref.loc[other_techs, "c_inv"]).abs() > 1e-9
    if diff_other.any():
        raise AssertionError(
            f"{target}: c_inv changed for non-ATB technologies: "
            f"{redeployed.loc[other_techs].index[diff_other].tolist()}")

    print(f"OK -- {target} cost/lifetime block redeployed, c_inv ratio verified for "
          f"{sorted(ratio_techs)}, unchanged for every other technology")


OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2050/no_transition/02_REF_REGION/Technologies.csv cost/lifetime block redeployed, c_inv ratio verified for ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], unchanged for every other technology
OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2050/early_access/02_REF_REGION/Technologies.csv cost/lifetime block redeployed, c_inv ratio verified for ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], unchanged for every other technology
OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2050/late_access/02_REF_REGION/Technologies.csv cost/lifetime block redeployed, c_inv ratio verified for ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], unchanged for every other technology
OK -- ../../../EnergyScope_BO_nord_amazonia/Data/2050/early_access_brazil/02_REF_REGION/Technologies.csv cost/lifetime block redeployed, c_inv ratio verified for ['BATT_HS', 'BATT_LI', 'PV_HS', 'PV_UTILITY'], unchanged for every other technology


## 10. Vintage-aware capacity registry — report

The full registry built in Section 0bis/setup (`REGISTRY`): every (scenario, cluster, technology)
row that carries a nonzero capacity at 2035, tagged by vintage year, `c_inv`, and whether it is
chained forward to 2050 or retired. Saved to
`output_energyscope_2050/vintage_registry.csv` for later system-cost reconstruction
post-processing (retired legacy capital is not simply "free" — it was paid for at its own vintage
year and should not double-count against the 2050 catalogue's frozen `c_inv`).


In [14]:
REGISTRY_OUT_PATH = "output_energyscope_2050/vintage_registry.csv"
os.makedirs(os.path.dirname(REGISTRY_OUT_PATH), exist_ok=True)
REGISTRY.to_csv(REGISTRY_OUT_PATH, sep=";", index=False)
print(f"Registry saved to {REGISTRY_OUT_PATH} ({len(REGISTRY)} rows)")
print()

# Summary counts
summary = REGISTRY.groupby(["scenario", "status"]).size().unstack(fill_value=0)
print("Row counts by scenario/status:")
print(summary)
print()

# PV_UTILITY / GENSET_DIESEL full detail -- the only technologies with a legacy/new-build split
print("=== PV_UTILITY / GENSET_DIESEL vintage detail (all scenarios/clusters with nonzero capacity) ===")
special = REGISTRY[REGISTRY["technology"].isin(["PV_UTILITY", "GENSET_DIESEL"])].sort_values(
    ["technology", "cluster", "scenario", "vintage_year"])
with pd.option_context("display.max_rows", None, "display.width", 140):
    print(special[["scenario", "cluster", "technology", "vintage_year", "capacity_GW",
                    "c_inv", "status", "f_min_2050_GW"]].to_string(index=False))
print()

# Non-survivor retirements with material capacity (top 10 by capacity, for a sanity read)
print("=== Largest non-survivor retirements (top 10 by capacity_GW) ===")
retired = REGISTRY[(REGISTRY["status"] == "retired_at_2050")
                    & (~REGISTRY["technology"].isin(["PV_UTILITY", "GENSET_DIESEL"]))]
top_retired = retired.sort_values("capacity_GW", ascending=False).head(10)
print(top_retired[["scenario", "cluster", "technology", "capacity_GW", "c_inv"]].to_string(index=False))


Registry saved to output_energyscope_2050/vintage_registry.csv (596 rows)

Row counts by scenario/status:
status               chained_to_2050  retired_at_2050
scenario                                             
early_access                      84               66
early_access_brazil               84               66
late_access                       84               66
no_transition                     90               56

=== PV_UTILITY / GENSET_DIESEL vintage detail (all scenarios/clusters with nonzero capacity) ===
           scenario cluster    technology vintage_year  capacity_GW      c_inv          status  f_min_2050_GW
       early_access      C1 GENSET_DIESEL         2035     0.000000 384.600000 chained_to_2050       0.000000
early_access_brazil      C1 GENSET_DIESEL         2035     0.000000 384.600000 chained_to_2050       0.000000
        late_access      C1 GENSET_DIESEL         2035     0.000000 384.600000 chained_to_2050       0.000000
      no_transition      C1 GENS

## 11. Assert deployed `f_min` matches the registry

Re-reads the **just-deployed** `Data/2050/{scenario}/C{k}/Technologies.csv` files written in
Section 9 (not the in-notebook `assembled` dataframes still in memory) and checks, for every
registry row, that the deployed `f_min` equals the registry's `f_min_2050_GW` -- closes the loop
between "what the registry says should be chained" and "what actually landed on disk".


In [15]:
DEPLOYED_TECH_PATH_TMPL = "../../../EnergyScope_BO_nord_amazonia/Data/2050/{scenario}/C{k}/Technologies.csv"

deployed_check_cache = {}
for scenario in DEPLOY_SCENARIOS_2050:
    deployed_check_cache[scenario] = {}
    for k in range(1, 6):
        d = pd.read_csv(DEPLOYED_TECH_PATH_TMPL.format(scenario=scenario, k=k), sep=";", index_col=0)
        d.index = d.index.str.strip()
        deployed_check_cache[scenario][k] = d

# Rule (e) sets f_min first (captured in REGISTRY); two later, independent, pre-existing rules
# unconditionally override it afterward for specific technologies, same ordering as every prior
# build of this notebook (Section 4): rule (b) always sets PV_HS f_min=0 regardless of what it
# chained to (legacy kits dead, new kits freely buildable -- not a vintage-carryover mechanism);
# rule (d) always sets STOVE_WOOD/STOVE_LPG f_min from the frozen Census 2024 figure regardless of
# their non-survivor classification under rule (e) (an a-priori sizing requirement, not a capacity
# carryover). These three are excluded from the strict equality check below for that documented
# reason -- not a bug.
OVERRIDDEN_DOWNSTREAM_OF_RULE_E = {"PV_HS", "STOVE_WOOD", "STOVE_LPG"}

# PV_UTILITY/GENSET_DIESEL carry TWO registry rows per (scenario, cluster) when legacy > 0: a
# 'retired_at_2050' row (the legacy share, f_min_2050_GW=0.0 by definition -- it does not claim
# anything about the actually-deployed f_min) and a 'chained_to_2050' row (the 2035-vintage
# new-build share, whose f_min_2050_GW IS what should have been deployed). Only 'chained_to_2050'
# rows are checked against the deployed value; 'retired_at_2050' rows are an audit trail, not a
# second, competing claim about the same deployed number.
registry_chained = REGISTRY[REGISTRY["status"] == "chained_to_2050"]

mismatches = []
skipped = []
for _, row in registry_chained.iterrows():
    scenario, cluster, tech = row["scenario"], row["cluster"], row["technology"]
    k = int(cluster[1:])
    deployed_fmin = float(deployed_check_cache[scenario][k].loc[tech, "f_min"])
    expected_fmin = row["f_min_2050_GW"]
    if tech in OVERRIDDEN_DOWNSTREAM_OF_RULE_E:
        skipped.append((scenario, cluster, tech))
        continue
    if abs(deployed_fmin - expected_fmin) > 1e-9:
        mismatches.append((scenario, cluster, tech, deployed_fmin, expected_fmin))

if mismatches:
    raise AssertionError(f"{len(mismatches)} deployed f_min values do not match the registry: {mismatches[:10]}")

print(f"ASSERT OK -- deployed f_min matches the registry for all {len(registry_chained) - len(skipped)} "
      f"chained registry rows not overridden downstream, {len(DEPLOY_SCENARIOS_2050)} scenarios x C1-C5")
print(f"({len(skipped)} rows skipped: PV_HS/STOVE_WOOD/STOVE_LPG, overridden by rules (b)/(d) "
      f"after rule (e) by design -- see comment above; "
      f"{(REGISTRY['status']=='retired_at_2050').sum()} retired_at_2050 rows not checked here, "
      f"by definition -- see comment above)")

# Also re-verify the special-legacy numbers directly against the just-reread deployed files
# (independent of clusters_out from Section 7, which reads output_energyscope_2050/ not Data/2050/),
# against the live CHAINED_FMIN_2050 -- not a hardcoded snapshot, since rule (e) depends on solved
# 2035 F values that shift whenever 2035 is resolved under different costs.
print()
print("=== Special-legacy f_min check, re-read directly from Data/2050/ (deployed) vs CHAINED_FMIN_2050 ===")
for scenario in DEPLOY_SCENARIOS_2050:
    for tech, k in SPECIAL_LEGACY_CHECK:
        expected = CHAINED_FMIN_2050[scenario][k][tech]
        actual = float(deployed_check_cache[scenario][k].loc[tech, "f_min"])
        assert abs(actual - expected) < 5e-6, (
            f"{scenario} C{k} {tech}: deployed f_min={actual}, expected {expected}")
        print(f"  {scenario:<14}C{k}  {tech:<16}{actual:>12.6f}  (expected {expected:>10.6f})  OK")
print("ASSERT OK -- all special-legacy f_min values confirmed on the actually-deployed Data/2050/ files")


ASSERT OK -- deployed f_min matches the registry for all 329 chained registry rows not overridden downstream, 4 scenarios x C1-C5


(13 rows skipped: PV_HS/STOVE_WOOD/STOVE_LPG, overridden by rules (b)/(d) after rule (e) by design -- see comment above; 254 retired_at_2050 rows not checked here, by definition -- see comment above)

=== Special-legacy f_min check, re-read directly from Data/2050/ (deployed) vs CHAINED_FMIN_2050 ===
  no_transition C3  GENSET_DIESEL       0.000000  (expected   0.000000)  OK
  no_transition C4  GENSET_DIESEL       0.000000  (expected   0.000000)  OK
  no_transition C5  GENSET_DIESEL       0.000000  (expected   0.000000)  OK
  no_transition C4  PV_UTILITY          0.000000  (expected   0.000000)  OK
  no_transition C5  PV_UTILITY          0.000000  (expected   0.000000)  OK
  early_access  C3  GENSET_DIESEL       0.000000  (expected   0.000000)  OK
  early_access  C4  GENSET_DIESEL       0.000000  (expected   0.000000)  OK
  early_access  C5  GENSET_DIESEL       0.000000  (expected   0.000000)  OK
  early_access  C4  PV_UTILITY          0.059047  (expected   0.059047)  OK
  early_acces

## 12. Reprint `reg_technologies.dat` from the deployed catalog (no solve)

For each of the 4 scenarios, instantiates `Esmc`, reads the just-deployed `Data/2050/{scenario}/`
catalog (`read_data_indep()` + `init_regions()`), applies the same pre-solve preprocessing
`scripts/run.py` applies for these case studies (the `ft_to_drop` biofuel-conversion exclusion,
unconditional; the `PV_UTILITY`/`BATT_LI` `f_max = 1e15` supply-side unlock for
`early_access`/`late_access` -- and, since `"norte_amazonia_early_access_brazil_2050"` also matches
the `early_access` prefix check in `scripts/run.py`, that unlock applies to `early_access_brazil`
too), reuses the shared typical-day cache (`algo='read'`, no AMPL call), then calls
`print_data(indep=True)` to (re)write `reg_technologies.dat` and friends into
`case_studies/C1_C2_C3_C4_C5/norte_amazonia_{scenario}_2050/`.

**This does NOT call `set_esom()` or `solve_esom()` -- no AMPL solve is launched.** The four 2050
case-study directories were completely empty before this cell ran (no prior `reg_*.dat`, no
`outputs/`); this is the first time they are populated, and only with the pre-solve `.dat` files.


In [16]:
import sys
sys.path.insert(0, r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
from pathlib import Path
from esmc import Esmc

REPRINT_YEAR = 2050
REPRINT_SCENARIO_NAME = {  # Esmc config 'scenario' value -> Data/2050/<scenario>/ subfolder
    "no_transition": "no_transition",
    "early_access": "early_access",
    "late_access": "late_access",
    "early_access_brazil": "early_access_brazil",
}
REPRINT_CASE_STUDY = {s: f"norte_amazonia_{s}_{REPRINT_YEAR}" for s in DEPLOY_SCENARIOS_2050}

# Mirrors scripts/run.py's ft_to_drop -- applied unconditionally for every case there.
FT_TO_DROP = ['BIOMASS_TO_GASOLINE', 'BIOMASS_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_DIESEL',
              'POWER_TO_GASOLINE', 'POWER_TO_DIESEL', 'H2_TO_GASOLINE', 'H2_TO_DIESEL']

AMPL_PATH = r'C:\Users\valen\AMPL'  # unused by algo='read' -- init_ta's signature requires it regardless

REPRINT_MODELS = {}
for scenario in DEPLOY_SCENARIOS_2050:
    case_study = REPRINT_CASE_STUDY[scenario]
    config = {'case_study': case_study, 'comment': 'vintage-aware f_min reprint (no solve)',
              'regions_names': ['C1', 'C2', 'C3', 'C4', 'C5'],
              'gwp_limit_overall': None, 're_share_primary': None, 'f_perc': True,
              'year': REPRINT_YEAR, 'scenario': REPRINT_SCENARIO_NAME[scenario]}

    my_model = Esmc(config, nbr_td=16)
    current_project = Path(r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
    my_model.project_dir = current_project
    my_model.dat_dir = current_project / 'case_studies' / my_model.space_id / '00_td_dat'
    my_model.cs_dir = current_project / 'case_studies' / my_model.space_id / case_study
    my_model.dat_dir.mkdir(parents=True, exist_ok=True)
    my_model.cs_dir.mkdir(parents=True, exist_ok=True)

    my_model.read_data_indep()
    my_model.init_regions()

    my_model.ref_region.data['Technologies'] = my_model.ref_region.data['Technologies'].drop(index=FT_TO_DROP)
    my_model.data_indep['Layers_in_out'] = my_model.data_indep['Layers_in_out'].drop(index=FT_TO_DROP)
    for r_code, region in my_model.regions.items():
        region.data['Technologies'] = region.data['Technologies'].drop(index=FT_TO_DROP)

    # early_access/late_access supply-side unlock (early_access_brazil's case_study string also
    # matches the 'norte_amazonia_early_access_' prefix check used in scripts/run.py)
    if case_study.startswith('norte_amazonia_early_access_') or case_study.startswith('norte_amazonia_late_access_'):
        for r_code, region in my_model.regions.items():
            region.data['Technologies'].loc['PV_UTILITY', 'f_max'] = 1e15
            region.data['Technologies'].loc['BATT_LI', 'f_max'] = 1e15

    # Pre-print sanity check: in-memory region data (freshly read from the deployed CSVs) must
    # match the registry for the two chained techs, before print_data() writes anything.
    for tech in ["PV_UTILITY", "GENSET_DIESEL"]:
        for k in range(1, 6):
            region_code = f"C{k}"
            if region_code not in my_model.regions or tech not in my_model.regions[region_code].data['Technologies'].index:
                continue
            in_memory_fmin = float(my_model.regions[region_code].data['Technologies'].loc[tech, 'f_min'])
            expected = deployed_check_cache[scenario][k].loc[tech, 'f_min']
            assert abs(in_memory_fmin - expected) < 1e-9, (
                f"{scenario} {region_code} {tech}: in-memory f_min={in_memory_fmin} after init_regions(), "
                f"expected {expected} (deployed CSV)")

    my_model.init_ta(algo='read', ampl_path=AMPL_PATH)  # reuses the shared 00_td_dat cache -- no AMPL call
    my_model.print_td_data()
    my_model.print_data(indep=True)

    REPRINT_MODELS[scenario] = my_model
    print(f"OK -- reg_technologies.dat reprinted for {case_study} at {my_model.cs_dir} (pre-print in-memory check passed, no solve)")


[INFO    ] (read_data_indep): Read indep data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\00_INDEP


[INFO    ] (init_regions): Initialising regions: C1, C2, C3, C4, C5


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\02_REF_REGION


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\C1


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\C2


...

[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_td_data): Printing TD data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050\reg_16TD.dat


[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_data): Printing regional data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050


OK -- reg_technologies.dat reprinted for norte_amazonia_early_access_brazil_2050 at C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050 (pre-print in-memory check passed, no solve)


### 12bis. Spot-check the reprinted `.dat` text itself

Reads `reg_technologies.dat` back as text and confirms the deployed `f_min` values for the two
chained technologies actually landed in the printed AMPL data file (not just in the in-memory
`Esmc` object checked above).


In [17]:
# reg_technologies.dat row layout (confirmed from a live printed file):
# REGION  TECH  c_inv  c_maint  gwp_constr  lifetime  c_p  fmin_perc  fmax_perc  f_min  f_max  f_min_prod  f_max_prod
# -> f_min is whitespace-split field index 9, f_max is field index 10 (0-based).
F_MIN_FIELD_INDEX = 9
F_MAX_FIELD_INDEX = 10

def _find_row(lines, k, tech):
    return next((l for l in lines if l.split() and l.split()[0] == f"C{k}" and l.split()[1] == tech), None)

def _printed_value(lines, k, tech, field_index):
    row = _find_row(lines, k, tech)
    assert row is not None, f"no row found for C{k} {tech}"
    return float(row.split()[field_index])

print("=== reg_technologies.dat text spot-check (parses actual f_min/f_max values, not just row presence) ===")
for scenario in DEPLOY_SCENARIOS_2050:
    dat_path = REPRINT_MODELS[scenario].cs_dir / "reg_technologies.dat"
    lines = dat_path.read_text(encoding="utf-8").splitlines()
    for k in [4, 5]:
        for tech in ["PV_UTILITY", "GENSET_DIESEL"]:
            expected_fmin = float(deployed_check_cache[scenario][k].loc[tech, "f_min"])
            printed_fmin = _printed_value(lines, k, tech, F_MIN_FIELD_INDEX)
            assert abs(printed_fmin - expected_fmin) < 1e-9, (
                f"{scenario} reg_technologies.dat C{k} {tech}: printed f_min={printed_fmin}, "
                f"expected {expected_fmin} (deployed CSV)")
    print(f"OK -- {dat_path}: PV_UTILITY/GENSET_DIESEL f_min in C4/C5 match the deployed catalog exactly")

# Rule (i) correction (extended 2026-08-05b), verified directly in the reprinted .dat text (not
# just the in-memory Esmc object or the deployed CSV): GENSET_DIESEL f_max must read as Infinity
# in every cluster, f_min must read as 0.0, in EVERY 2050 scenario.
print()
print("=== reg_technologies.dat text spot-check -- rule (i) correction (GENSET_DIESEL, all scenarios) ===")
for scenario in DEPLOY_SCENARIOS_2050:
    dat_path = REPRINT_MODELS[scenario].cs_dir / "reg_technologies.dat"
    lines = dat_path.read_text(encoding="utf-8").splitlines()
    for k in range(1, 6):
        row = _find_row(lines, k, "GENSET_DIESEL")
        assert row is not None, f"{scenario} reg_technologies.dat: no row found for C{k} GENSET_DIESEL"
        fields = row.split()
        printed_fmin = float(fields[F_MIN_FIELD_INDEX])
        printed_fmax_raw = fields[F_MAX_FIELD_INDEX]
        assert printed_fmin == 0.0, f"{scenario} C{k} GENSET_DIESEL: printed f_min={printed_fmin}, expected 0.0"
        assert printed_fmax_raw == "Infinity", (
            f"{scenario} C{k} GENSET_DIESEL: printed f_max={printed_fmax_raw}, expected 'Infinity' "
            f"(rule (i) correction did not make it into the reprinted .dat)")
    print(f"OK -- {dat_path}: GENSET_DIESEL f_min=0.0 / f_max=Infinity in every cluster (rule (i) confirmed in the printed .dat)")

print()
print("Reprint complete for all 4 scenarios. No set_esom()/solve_esom() call was made -- "
      "no AMPL solve was launched, per instructions.")


=== reg_technologies.dat text spot-check (parses actual f_min/f_max values, not just row presence) ===
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_no_transition_2050\reg_technologies.dat: PV_UTILITY/GENSET_DIESEL f_min in C4/C5 match the deployed catalog exactly
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_2050\reg_technologies.dat: PV_UTILITY/GENSET_DIESEL f_min in C4/C5 match the deployed catalog exactly
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_late_access_2050\reg_technologies.dat: PV_UTILITY/GENSET_DIESEL f_min in C4/C5 match the deployed catalog exactly
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050\reg_technologies.dat: PV_UTILITY/GENSET_DIESEL f_min in C4/C5 match the deployed catalog exactly

=== reg_technologies.dat text spot-check -- rule (i) correction (